# BERT — AdamW · Test Multi-Seeds (5 seeds)
**Objectif :** Prouver que la performance d'AdamW est statistiquement robuste et non due au hasard.  
**Dataset :** SST-2 (GLUE) · **Modèle :** bert-base-uncased · **Seeds :** 42, 123, 2024, 7, 999

In [ ]:
# Cell 0 — Installation des dépendances
!pip install transformers datasets torch scikit-learn scipy -q

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import time
from scipy import stats
from sklearn.metrics import f1_score, accuracy_score

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed
)
from torch.optim import AdamW
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 2 — Configuration
MODEL_NAME   = 'bert-base-uncased'
SEEDS        = [42, 123, 2024, 7, 999]
LR           = 2e-5
BATCH_SIZE   = 16
NUM_EPOCHS   = 2
MAX_LENGTH   = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0

print('Configuration AdamW — Multi-Seeds')
print(f'  Seeds        : {SEEDS}')
print(f'  lr           : {LR}')
print(f'  batch_size   : {BATCH_SIZE}')
print(f'  epochs       : {NUM_EPOCHS}')
print(f'  weight_decay : {WEIGHT_DECAY}')
print(f'  Runs totales : {len(SEEDS)}')

In [ ]:
# Cell 3 — Dataset + Tokenisation (fait une seule fois)
dataset   = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(['sentence', 'idx'])
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch')

collator       = DataCollatorWithPadding(tokenizer)
train_dataset  = tokenized['train']
val_dataset    = tokenized['validation']

val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collator)

print(f'Train : {len(train_dataset)} exemples')
print(f'Val   : {len(val_dataset)} exemples')

In [ ]:
# Cell 4 — Fonctions utilitaires
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds  = logits.argmax(dim=-1).cpu().tolist()
            labels = batch['labels'].cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels)
    acc = accuracy_score(all_labels, all_preds) * 100
    f1  = f1_score(all_labels, all_preds, average='weighted') * 100
    return acc, f1

print('Fonctions prêtes.')

In [ ]:
# Cell 5 — Boucle Multi-Seeds AdamW

# Config redéfinie ici pour éviter les erreurs si une cellule est sautée
MODEL_NAME   = 'bert-base-uncased'
SEEDS        = [42, 123, 2024, 7, 999]
LR           = 2e-5
BATCH_SIZE   = 16
NUM_EPOCHS   = 2
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
SEP          = '=' * 50

results = []  # [{seed, acc, f1, time_s}]

for seed in SEEDS:
    print('\n' + SEP)
    print('Seed ' + str(seed) + ' — AdamW')
    print(SEP)
    set_all_seeds(seed)

    # Modèle réinitialisé depuis les poids pré-entraînés à chaque seed
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collator
    )

    optimizer = AdamW(
        model.parameters(),
        lr=LR,
        eps=1e-8,
        weight_decay=WEIGHT_DECAY
    )

    num_steps  = len(train_loader) * NUM_EPOCHS
    num_warmup = int(num_steps * WARMUP_RATIO)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup, num_steps
    )

    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        for step, batch in enumerate(train_loader):
            batch  = {k: v.to(device) for k, v in batch.items()}
            loss   = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc, val_f1 = evaluate(model, val_loader)
        elapsed = time.time() - t0
        print('  Epoch ' + str(epoch+1) + '/' + str(NUM_EPOCHS) +
              ' | loss=' + str(round(avg_loss, 4)) +
              ' | val_acc=' + str(round(val_acc, 2)) + '%' +
              ' | f1=' + str(round(val_f1, 2)) + '%' +
              ' | ' + str(int(elapsed)) + 's')

    total_time = time.time() - t0
    final_acc, final_f1 = evaluate(model, val_loader)
    results.append({'seed': seed, 'acc': final_acc, 'f1': final_f1, 'time_s': total_time})
    print('\n  Résultat final seed=' + str(seed) +
          ' → acc=' + str(round(final_acc, 2)) + '%' +
          ' | f1=' + str(round(final_f1, 2)) + '%' +
          ' | temps=' + str(int(total_time)) + 's')

print('\nToutes les seeds terminées !')


In [ ]:
# Cell 6 — Statistiques
accs  = [r['acc']  for r in results]
f1s   = [r['f1']   for r in results]
times = [r['time_s'] for r in results]

def bootstrap_ci(data, n_boot=2000, ci=0.95):
    boots = [np.mean(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)]
    lo = ((1-ci)/2)*100
    hi = (1-(1-ci)/2)*100
    return np.percentile(boots, [lo, hi])

ci = bootstrap_ci(accs)

print('='*55)
print('RÉSULTATS STATISTIQUES — AdamW · BERT · SST-2')
print('='*55)
print(f'  Runs          : {len(SEEDS)} seeds')
print(f'  Accuracy mean : {np.mean(accs):.2f}%')
print(f'  Accuracy std  : ±{np.std(accs, ddof=1):.2f}%')
print(f'  Accuracy min  : {np.min(accs):.2f}%')
print(f'  Accuracy max  : {np.max(accs):.2f}%')
print(f'  IC 95%        : [{ci[0]:.2f}%, {ci[1]:.2f}%]')
print(f'  F1 mean       : {np.mean(f1s):.2f}% ±{np.std(f1s, ddof=1):.2f}')
print(f'  Temps moyen   : {np.mean(times)/60:.1f} min/run')
print()

df = pd.DataFrame(results)
df['optimiseur'] = 'AdamW'
df.to_csv('adamw_multiseed_results.csv', index=False)
print('Résultats sauvegardés : adamw_multiseed_results.csv')
print(df[['seed','acc','f1','time_s']].to_string(index=False))

In [ ]:
# Cell 7 — Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('AdamW — BERT — SST-2 · Validation Multi-Seeds (n=5)', fontsize=14, fontweight='bold')

# Plot 1 : accuracy par seed
ax = axes[0]
colors = ['#7F77DD' if a == max(accs) else '#AAAADD' for a in accs]
bars = ax.bar([f'seed\n{r["seed"]}' for r in results], accs, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(np.mean(accs), color='#3C3489', linestyle='--', linewidth=1.8, label=f'mean = {np.mean(accs):.2f}%')
ax.fill_between(range(len(SEEDS)), ci[0], ci[1], alpha=0.12, color='#7F77DD', label='IC 95%')
for bar, v in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, f'{v:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(min(accs)-2, max(accs)+2)
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Accuracy par seed')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Plot 2 : distribution + stats
ax2 = axes[1]
ax2.violinplot([accs], positions=[1], showmeans=True, showmedians=True)
ax2.scatter([1]*len(accs), accs, color='#7F77DD', zorder=5, s=60)
ax2.set_xticks([1])
ax2.set_xticklabels(['AdamW'])
ax2.set_ylabel('Validation Accuracy (%)')
ax2.set_title(f'Distribution — mean={np.mean(accs):.2f}% ±{np.std(accs, ddof=1):.2f}')
ax2.grid(axis='y', alpha=0.3)

textstr = f'mean = {np.mean(accs):.2f}%\nstd  = ±{np.std(accs, ddof=1):.2f}%\nIC95 = [{ci[0]:.2f}, {ci[1]:.2f}]'
ax2.text(0.05, 0.05, textstr, transform=ax2.transAxes, fontsize=9,
         verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='#EEEDFE', alpha=0.8))

plt.tight_layout()
plt.savefig('adamw_multiseed_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : adamw_multiseed_results.png')

In [ ]:
# Cell 8 — Résumé final
print('='*60)
print('RÉSUMÉ — AdamW · BERT · SST-2 · Multi-Seeds')
print('='*60)
print(f'  Optimiseur    : AdamW (lr={LR}, wd={WEIGHT_DECAY}, eps=1e-8)')
print(f'  Modèle        : {MODEL_NAME} (109M paramètres)')
print(f'  Dataset       : SST-2 — {len(train_dataset)} train / {len(val_dataset)} val')
print(f'  Seeds testées : {SEEDS}')
print()
print(f'  Accuracy      : {np.mean(accs):.2f}% ± {np.std(accs, ddof=1):.2f}%')
print(f'  IC 95%        : [{ci[0]:.2f}%, {ci[1]:.2f}%]')
print(f'  F1 score      : {np.mean(f1s):.2f}% ± {np.std(f1s, ddof=1):.2f}%')
print()
print('  → Résultats à comparer avec sgd-multiseed-test.ipynb')
print('    pour le test de Welch (significativité statistique)')